In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from DeepScence.api import DeepScence
from SenCID.api import SenCID
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from dca.api import dca
import os
os.chdir(b'/Users/lele/Library/Mobile Documents/com~apple~CloudDocs/Research/Aging')

/Users/lele/Downloads/anaconda3/envs/sene/lib/python3.8/site-packages/kopt/config.py:60: YAMLLoadWarning: calling yaml.load() without Loader=... is deprecated, as the default Loader is unsafe. Please read https://msg.pyyaml.org/load for full details.
  _config = yaml.load(open(_config_path))


In [ ]:
# st = sc.read_h5ad("./data/additional_sc/stereo-seq/micro.h5ad")
# counts_df = pd.DataFrame(st.X.toarray(), index=st.obs_names, columns=st.var_names)
# meta_df = st.obs
# counts_df.to_csv("./data/additional_sc/stereo-seq/counts.csv")
# meta_df.to_csv("./data/additional_sc/stereo-seq/meta.csv")

### read datasets and gs

In [2]:
datasets = ["mouse_notexin_d2", "mouse_notexin_d5", "human_micro", "mouse_ctx", "mouse_aging"]
all_gs = pd.read_csv("./data/coreGS_v2.csv", index_col=0)
columns_to_use = all_gs.columns[:9]
anchors = {
    "trans": ["STAT1", "Stat1"],
    "network": ["IL6", "Il6"],
    "sensig": [None, None],
    "Senmayo": [None, None],
    "geneAge": [None, None],
    "cellAge": [None, None],
    "CSgene": [None, None],
    "SASP": ["IL6", "Il6"],
    "Quest": [None, None]
}
gs_list_human = {col: all_gs.index[all_gs[col] == True].tolist() for col in columns_to_use}
gs_list_mouse = {col: all_gs["mouse_gene"][all_gs[col] == True].tolist() for col in columns_to_use}

### get all scores

In [4]:
args = {
        "binarize": False,
        "verbose": False
    }
for d in datasets:
    adata = sc.read_h5ad(f"./data/VADLIATION_DATA/SPATIAL/Current/h5ad/{d}.h5ad")
    
    m = pd.DataFrame(index=adata.obs_names)

    if d.startswith("mouse_"):
        gs_list = gs_list_mouse
        species = "mouse"
        use = 1
    else:
        gs_list = gs_list_human
        species = "human"
        use = 0

    # DeepScence + all gs, run for each cell type
    for gsname, gs in gs_list.items():
        print(f"Running {d} : DeepScence + {gsname}")
        adata = DeepScence(adata, custome_gs = gs, anchor_gene=anchors[gsname][use], **args)
        m.loc[adata.obs_names, f"ds_{gsname}"] = adata.obs["ds"].values

    # DeepScence + different n
    for n in [3,4,5,6]:
        print(f"Running {d}: DeepScence + >={n}...")
        adata = DeepScence(adata, n=n, species=species, **args)
        m.loc[adata.obs_names, f"ds_{n}+"] = adata.obs["ds"].values

    # SenCID

    # to run SenCID, we need to substitute sencid genes into human homologs
    if d.startswith("mouse_"):
        adata.var['gene_symbols'] = adata.var.index
        c = pd.read_csv("./data/in_vivo/gene_convert.csv")
        gene_map = dict(zip(c['original'].dropna(), c['converted'].dropna()))
        adata.var["converted_gene"] = adata.var["gene_symbols"].map(lambda x: gene_map.get(x, x))
        adata.var_names = adata.var["converted_gene"].values
        adata.var_names_make_unique()
    
    
    pred_dict, recSID, tmpfiles = SenCID(
                adata=adata,
                sidnums=[1, 2, 3, 4, 5, 6],
                denoising=False,
                binarize=True,
                threads=1,
                savetmp=True,
            )
    binary2 = []
    scores2 = []
    for i in range(len(recSID)):
        rec = recSID["RecSID"].iloc[i]
        score = pred_dict[rec]["SID_Score"].iloc[i]
        b = pred_dict[rec]["Binarization"].iloc[i]
        binary2.append(b)
        scores2.append(score)
    m["SID_binary"] = binary2
    m["SID_score"] = scores2
    
    # save
    m.to_csv(f"./data/VADLIATION_DATA/SPATIAL/Current/metas/{d}_scores_part2.csv")

[2025-03-12 21:54] Input is not count, processed 15042 genes and 859 cells.
[2025-03-12 21:54] Using 38 genes in the gene set for scoring.
[2025-03-12 21:54] Lambda provided, capturing scores in 2 neurons.
[2025-03-12 21:54] Training on 774 cells, validate on 85 cells.


Running mouse_notexin_d2 : DeepScence + trans


 58%|███████████████████████▉                 | 175/300 [00:02<00:02, 60.94it/s]
[2025-03-12 21:54] Input is not count, processed 15042 genes and 859 cells.
[2025-03-12 21:54] Using 32 genes in the gene set for scoring.
[2025-03-12 21:54] Lambda provided, capturing scores in 2 neurons.
[2025-03-12 21:54] Training on 774 cells, validate on 85 cells.


Running mouse_notexin_d2 : DeepScence + network


100%|█████████████████████████████████████████| 300/300 [00:04<00:00, 72.51it/s]
[2025-03-12 21:54] Input is not count, processed 15042 genes and 859 cells.
[2025-03-12 21:54] Using 1083 genes in the gene set for scoring.
[2025-03-12 21:54] Lambda provided, capturing scores in 2 neurons.
[2025-03-12 21:54] Training on 774 cells, validate on 85 cells.


...

Running mouse_aging : DeepScence + cellAge


[2025-03-12 22:31] Input is not count, processed 18986 genes and 23488 cells.
[2025-03-12 22:31] Using 810 genes in the gene set for scoring.
[2025-03-12 22:31] Lambda provided, capturing scores in 2 neurons.
[2025-03-12 22:31] Training on 21140 cells, validate on 2348 cells.
100%|█████████████████████████████████████████| 300/300 [08:37<00:00,  1.72s/it]


Running mouse_aging : DeepScence + CSgene


[2025-03-12 22:40] Input is not count, processed 18986 genes and 23488 cells.
[2025-03-12 22:40] Using 384 genes in the gene set for scoring.
[2025-03-12 22:41] Lambda provided, capturing scores in 2 neurons.
[2025-03-12 22:41] Training on 21140 cells, validate on 2348 cells.
 28%|███████████▊                              | 84/300 [01:23<03:34,  1.01it/s]


KeyboardInterrupt: 